## final_project_part_2.ipynb
### DSA4323/5323
### University of Oklahoma
This notebook is the second of two (or three for graduate students) of the final project for Ethics and Compliance in AI at the University of Oklahoma. In this assignment, you will analyze a dataset containing credit card default information. The dataset was obtained from the UCI Machine Learning Repository: https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients. It was initially used for the study Yeh, I., & Lien, C. (2009). The comparisons of data mining techniques for the predictive accuracy of probability of default of credit card clients. Expert Syst. Appl., 36, 2473-2480.

The variables in the dataset are:
- ID: unique identifier
- LIMIT_BAL: amount of credit given in NT dollars
- SEX: gender of the client (1=Male, 2=Female)
- EDUCATION: education level (1=Graduate School, 2=University, 3=High School, 4=Others, 5=Unknown, 6=Unknown)
- MARRIAGE: marital status (1=Married, 2=Single, 3=Others)
- AGE: age in of client in years
- PAY_0 to PAY_6: monthly repayment status from September 2005 to April 2005 in reverse order. Note that there is no PAY_1. This is a naming mistake by the authors. (-2=No consumption, -1=Paid in full, 0=Minimum payment, 1-8=Payment delay for x months, 9=Payment delay for nine or more months)
- BILL_AMT1 to BILL_AMT6: amount of bill statement in NT dollars from September 2005 to April 2005 in reverse order
- PAY_AMT1 to PAY_AMT6: amount of previous payment in NT dollars from September 2005 to April 2005 in reverse order
- default payment next month: whether the client made the next month payment - this is the label variable (1=Defaulted, 0=No Default)

In this notebook, _final\_project\_part\_2.ipynb_, you will read in the imputed dataset you saved in Part 1. You will prepare the data for modeling. You will then run a baseline logistic regression analysis, examine the results, and check fairness metrics. You will then repeat this procedure after using two pre-processing techniques that weight the data. You will conclude by creating a final model that uses intrinsic mitigation through fairness constraint and again examine results and check fairness metrics.

### Import libraries

Import the libraries required for this notebook and set random seeds to stabilize results.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from scipy.optimize import minimize
from scipy.special import expit

import os, random

# Set global seeds for reproducibility
os.environ["PYTHONHASHSEED"] = "42"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
random.seed(42)
np.random.seed(42)


### Fairness Metrics

You will use the function below to conduct fairness metrics. Make sure you understand the inputs and outputs of the function.

In [ ]:
# Define the fairness evaluation function

def check_fairness(y_true, y_pred, sensitive_attr):
    """
    Evaluates a model's fairness by comparing performance across different groups.
    
    Parameters:
    y_true : Series/Array - The actual ground truth labels (0 or 1)
    y_pred : Series/Array - The model's binary predictions (0 or 1)
    sensitive_attr : Series/Array - The binary sensitive feature (e.g., Sex: 1=Male, 0=Female)
    """
    
    # Create a temporary dataframe to align predictions with actuals and sensitive labels
    results = pd.DataFrame({'y': y_true, 'y_hat': y_pred, 'A': sensitive_attr})
    
    # Split the results into two groups based on the sensitive attribute
    # Group 0 is often the historically 'unprivileged' or 'protected' group
    g0 = results[results['A'] == 0] 
    g1 = results[results['A'] == 1] 
    
    # Dictionary to store calculated metrics for each group
    stats = {}
    
    for name, group in [('Female', g0), ('Male', g1)]:
        # Calculate the Confusion Matrix components:
        # TN: True Negatives, FP: False Positives, FN: False Negatives, TP: True Positives
        # labels=[0,1] ensures the matrix is always 2x2 even if a group is missing a class
        tn, fp, fn, tp = confusion_matrix(group['y'], group['y_hat'], labels=[0, 1]).ravel()
        
        # 1. Demographic Parity (Statistical Independence)
        # Ratio of positive predictions regardless of the actual ground truth.
        # Ideally, P(Y_hat=1 | A=0) == P(Y_hat=1 | A=1)
        dp = group['y_hat'].mean()
        
        # 2. True Positive Rate (TPR / Sensitivity): Part of Error Rate Parity/Separation
        # How often does the model correctly predict 'High Income' for those who actually have it?
        # A gap here indicates the model 'misses' qualified candidates in one group more than the other.
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        
        # 3. True Negative Rate (TNR / Specificity): Part of Error Rate Parity/Separation
        # How often does the model correctly predict 'Low Income' for those who actually have it?
        # A gap here indicates one group is being 'falsely accused' of being high-income more often.
        tnr = tn / (tn + fp) if (tn + fp) > 0 else 0

        # 4. False Positive Rate (FPR / Type I Error): Part of Error Rate Parity/Separation
        # How often does the model incorrectly predict 'High Income' for those who actually have it?
        # A gap here indicates one group is being 'falsely accused' of being high-income more often.
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

        # 5. Predictive Parity (Sufficiency)
        # If the model predicts 'High Income', how likely is it to be correct?
        # This checks if the 'meaning' of a positive prediction is the same across groups.
        pp = tp / (tp + fp) if (tp + fp) > 0 else 0
        
        # Store metrics for this group
        stats[name] = {'DP': dp, 'TPR': tpr, 'TNR': tnr, 'FPR': fpr, 'PP': pp}

    # --- Print Results Table ---
    print(f"{'Metric':<22} | {'Female':<8} | {'Male':<8} | {'Difference':<10} | {'% Difference':<12}")
    print("-" * 65)
    
    # Iterate through the metrics to calculate and display the absolute difference
    for m in ['DP', 'TPR', 'TNR', 'FPR', 'PP']:
        # Rename keys for the final display
        display_names = {
            'DP': 'Demographic Parity',
            'TPR': 'True Positive Rate',
            'TNR': 'True Negative Rate',
            'FPR': 'False Positive Rate',
            'PP': 'Predictive Parity'
        }
        
        f_val = stats['Female'][m]
        m_val = stats['Male'][m]
        diff = abs(f_val - m_val)
        pdiff = abs(f_val - m_val)/f_val*100 if m_val > 0 else 0
        
        print(f"{display_names[m]:<22} | {f_val:.3f}   | {m_val:.3f}   | {diff:.3f}   | {pdiff:.1f}%")
        
    print("-" * 65)
    print("Note: 'Error Rate Parity' is satisfied if both TPR and FPR differences are near zero.")

### Reweighting Parameters

The function below calculates the weights for two pre-processing methods:  Inverse Prevalence and Kamiran-Calders reweighting. You will call this function later in the script.

In [ ]:
def calculate_fairness_weights(A, y):
    """
    Calculates both Inverse Prevalence and Kamiran-Calders weights.
    Returns two series of weights aligned with the input indices.
    """
    df_temp = pd.DataFrame({"A": A, "Y": y})
    n = len(df_temp)
    
    # 1. Pre-calculate the probabilities needed
    # counts[(a, y)]
    counts = df_temp.groupby(["A", "Y"]).size()
    
    # P(A) and P(Y)
    p_a = df_temp["A"].value_counts(normalize=True)
    p_y = df_temp["Y"].value_counts(normalize=True)
    
    # P(A, Y)
    p_ay = counts / n

    # 2. Calculate weights using vectorized mapping (much faster than .apply)
    # Inverse Prevalence: 1 / Count(a, y)
    weights_inv = df_temp.apply(lambda r: 1.0 / counts[r["A"], r["Y"]], axis=1)
    
    # Kamiran-Calders: [P(A) * P(Y)] / P(A, Y)
    weights_kc = df_temp.apply(lambda r: (p_a[r["A"]] * p_y[r["Y"]]) / p_ay[r["A"], r["Y"]], axis=1)
    
    return weights_inv, weights_kc

# Usage:
# train_weights_inv, train_weights_kc = calculate_fairness_weights(A_train, y_train)

### Read in the data

Read in the data from the file you created in _final\_project\_part\_1.ipynb_. We then setup groups of the features for scaling and modeling.

In [ ]:
# Read in the imputed dataset
df = pd.read_csv("imputed_credit_card_data.csv")

# Set variable groups
bill_cols = [f'BILL_AMT{i}' for i in range(1, 7)]
pay_cols = [f'PAY_AMT{i}' for i in range(1, 7)]
sensitive_vars = ['SEX']
categorical_vars = ['EDUCATION', 'MARRIAGE']
ordinal_vars = [f'PAY_{i}' for i in range(1, 7)]
numerical_vars = ['LIMIT_BAL', 'AGE'] + bill_cols + pay_cols

one_hot_vars = ['EDUCATION', 'MARRIAGE']
raw_vars = ['SEX']
scale_vars = ['LIMIT_BAL', 'AGE'] + bill_cols + pay_cols + ordinal_vars



### Transform the variables

Transform the variables for modeling. You will one-hot encode the categorical variables and leave SEX and DEFAULTED alone (they are both 0/1 binary variables already). Note that scaling happens _after_ the train and test split to prevent data leakage.

In [ ]:
# Create One-Hot Encodings for categories
# drop_first=True prevents the 'dummy variable trap' (multi-collinearity)
X_cat = pd.get_dummies(df[one_hot_vars], drop_first=True)

# Extract Numeric features
X_num = df[scale_vars]

# Create the full feature set
X_combined = pd.concat([X_num, X_cat, df[raw_vars]], axis=1).astype(float)
A = df["SEX"]
y = df["DEFAULTED"]

# Split the data BEFORE scaling to prevent data leakage
X_train, X_test, \
y_train, y_test, \
A_train, A_test = train_test_split(
    X_combined, y, A, test_size=0.3, random_state=42
)   



### Transform variables (continued)

Z-score normalize the numerical variables.

In [ ]:
# SELECTIVE SCALING
# We only want to scale the columns that were originally continuous
scaler = StandardScaler()

# Scaling Training Data
X_train_scaled = X_train.copy()
X_train_scaled[scale_vars] = scaler.fit_transform(X_train[scale_vars])

# Scaling Testing Data
X_test_scaled = X_test.copy()
X_test_scaled[scale_vars] = scaler.transform(X_test[scale_vars])

# Note: All binary columns (Work, Marital, Race, Sex) remain exactly 0 or 1.

### Build the baseline model

Create a logistic regression model called _baseline_. Use sklearn LogisticRegression() and fit the model. Run it for a maximum of 500 iterations. Store the predictions from the test dataset (_X\_test\_scaled_) into an array named _y\_pred\_base_. Store the probabilities for the test dataset into an array named _y\_prob\_base_.

In [ ]:
# Train Baseline

# ENTER YOUR CODE HERE:


# END HERE


In [ ]:
# Save y, y_pred, y_prob, and A for post-processing
results_df = pd.DataFrame({
    'y_true': y_test,
    'y_pred_base': y_pred_base,
    'y_prob_base': y_prob_base,
    'A': A_test
})
results_df.to_csv("model_results_for_postprocessing.csv", index=False)

In [ ]:
# Print out the results for the baseline model (Unconstrained)
print("=== BASELINE RESULTS (Unconstrained) ===")
print("Classification Report\n", classification_report(y_test, y_pred_base))
print("Fairness Metrics")
check_fairness(y_test, y_pred_base, A_test)

### Create the weights

Call the function _calculate\_fairness\_weights() with the training split for _A_ and _y_. Save the training weights into two arrays called _weights\_inv\_train_ and _weights\_kc\_train_. These are for the two reweighting methods.

In [ ]:
# Calculate fairness weights for the training set

# BEGIN YOUR CODE HERE:


# END HERE

### Build the Inverse Prevalence Reweighted model

Create a logistic regression model called _inverse_. Use sklearn LogisticRegression() and fit the model. Run it for a maximum of 500 iterations. Include _weights\_inv\_train_ as sample weights. Store the predictions from the test dataset (_X\_test\_scaled_) into an array named _y\_pred\_inv_. Store the probabilities for the test dataset into an array named _y\_prob\_inv_.

In [ ]:
# Train Inverse Prevalence Reweighting

# BEGIN YOUR CODE HERE:


# END HERE


In [ ]:
# Print out the results for the Inverse Prevalence Reweighting model
print("=== INVERSE PREVALENCE REWEIGHTING RESULTS ===")
print("Classification Report\n", classification_report(y_test, y_pred_inv))
print("Fairness Metrics")
check_fairness(y_test, y_pred_inv, A_test)

### Build the Kamiran-Calders Reweighted model

Create a logistic regression model called _kcmod_. Use sklearn LogisticRegression() and fit the model. Run it for a maximum of 500 iterations. Include _weights\_kc\_train_ as sample weights. Store the predictions from the test dataset (_X\_test\_scaled_) into an array named _y\_pred\_kc_. Store the probabilities for the test dataset into an array named _y\_prob\_kc_.

In [ ]:
# Train Kamiran–Calders Reweighting

# BEGIN YOUR CODE HERE:


# END HERE


In [ ]:
# Print out the results for the Kamiran–Calders Reweighting model
print("=== KAMIRAN-CALDERS REWEIGHTING RESULTS ===")
print("Classification Report\n", classification_report(y_test, y_pred_kc))

print("Fairness Metrics")
check_fairness(y_test, y_pred_kc, A_test)


### Build Logistic Regression with Fairness Constraints model

Create a logistic regression model with the covariance constraint. The code for this next section can be copied directly from the assignment notebook _1\_logreg\_fairness\_answer.ipynb_, code cell 5 (assuming you didn't change them). You will build a model called _res_ and save your probabilities and predictions from the test data into two arrays called _fair\_prob_ and _fair\_pred_.

In [ ]:
# --- Run model with fairness constraints ---

# BEGIN YOUR CODE HERE:


# END HERE


In [ ]:
# Print out the results for the Fairness Constraint model
print("=== FAIRNESS CONSTRAINT RESULTS ===")
print("Classification Report\n", classification_report(y_test, fair_pred))

print("Fairness Metrics")
check_fairness(y_test, fair_pred, A_test)